In [ ]:
def vsftpd_log_set(ip: str, root_password: str):

    config_lines="""
dual_log_enable=YES
vsftpd_log_file=/var/log/vsftpd.log
"""
    COMMANDS =[
         ("vsftpd.conf 파일 설정", f"echo -e '{config_lines}' >> /etc/vsftpd/vsftpd.conf"),
         ("vsftpd.conf 재시작", "systemctl restart vsftpd"),
     ]

    run_cmd(get_ssh_client(ip, root_password, "root", 22), COMMANDS)

In [ ]:
import re

def parse_and_print_audit_log(log_line: str):

    base_pattern = re.search(
        r'^(?P<time>\w{3} \w{3}\s+\d+ \d+:\d+:\d+ \d+) \[(?P<pid>pid \d+)\] (?:\[(?P<user>\w*)\] )?(?P<msg>.*)',
        log_line
    )

    if not base_pattern:
        return  # 매칭되지 않는 빈 줄 등은 패스

    raw_time = base_pattern.group('time')
    pid = base_pattern.group('pid')

    # 대괄호가 없어서 user 그룹이 잡히지 않으면 None이 반환되므로 예외 처리
    account = base_pattern.group('user')
    if not account:
        account = "N/A"

    msg = base_pattern.group('msg')

    # 변수 초기화
    action = "UNKNOWN"
    status = "FAIL"
    ip_address = "UNKNOWN"
    target_file = "N/A"
    file_size = "N/A"
    xfer_speed = "N/A"

    # 행위 유형별 정밀 매칭
    if "CONNECT:" in msg:
        action = "CONNECT"
        status = "SUCCESS"
        ip_match = re.search(r'Client "([^"]+)"', msg)
        if ip_match: ip_address = ip_match.group(1).split(':')[-1]

    elif "LOGIN:" in msg:
        action = "LOGIN"
        status = "SUCCESS" if "OK LOGIN" in msg else "FAIL"
        ip_match = re.search(r'Client "([^"]+)"', msg)
        if ip_match: ip_address = ip_match.group(1).split(':')[-1]

    elif "UPLOAD:" in msg or "DOWNLOAD:" in msg:
        action = "UPLOAD" if "UPLOAD:" in msg else "DOWNLOAD"
        status = "SUCCESS" if "OK" in msg else "FAIL"
        xfer_pattern = re.search(r'Client "([^"]+)", "([^"]+)", (\d+) bytes, ([^\s]+)', msg)
        if xfer_pattern:
            ip_address = xfer_pattern.group(1).split(':')[-1]
            target_file = xfer_pattern.group(2)
            file_size = f"{int(xfer_pattern.group(3)):,} bytes"
            xfer_speed = xfer_pattern.group(4).replace('Kbyte/sec', ' Kbyte/sec')

    elif "DELETE:" in msg or "MKDIR:" in msg or "RMDIR:" in msg:
        if "DELETE:" in msg: action = "DELETE"
        elif "MKDIR:" in msg: action = "MKDIR"
        else: action = "RMDIR"
        status = "SUCCESS" if "OK" in msg else "FAIL"
        file_pattern = re.search(r'Client "([^"]+)", "([^"]+)"', msg)
        if file_pattern:
            ip_address = file_pattern.group(1).split(':')[-1]
            target_file = file_pattern.group(2)
    else:
        return

    # 가독성 높은 감사 포맷으로 출력
    print("  ============================================================")
    print("  [FTP ACCESS AUDIT LOG]")
    print("  ============================================================")
    print(f"  TIME        : {raw_time}")
    print(f"  ACCOUNT     : {account}")
    print(f"  IP ADDRESS  : {ip_address}")
    print(f"  PROCESS ID  : {pid}")
    print("  ------------------------------------------------------------")
    print(f"  ACTION      : {action}")
    print(f"  STATUS      : {status}")
    print("  ------------------------------------------------------------")
    print(f"  TARGET FILE : {target_file}")
    print(f"  FILE SIZE   : {file_size}")
    print(f"  XFER SPEED  : {xfer_speed}")
    print("  ============================================================\n")


def run_cmd_for_audit(client, cmd):
    """타입 힌트를 완전히 제거하고 변수만 남긴 올바른 코드"""
    for desc, com in cmd:
        print(f"\n[{desc}] 시작...")

        # 이제 위에서 받은 client를 사용하여 안전하게 명령어를 실행합니다.
        stdin, stdout, stderr = client.exec_command(com)

        for line in iter(stdout.readline, ""):
            parse_and_print_audit_log(line.strip())

        error = stderr.read().decode("utf-8")
        if error:
            print(f"\n[시스템 메시지]:\n{error}")

def print_all_saved_logs(ip: str, password: str):
    """기존에 누적된 vsftpd.log 전체를 가져와 분석을 실행하는 메인 함수입니다."""

    # cat 명령어를 튜플 리스트 구조로 생성
    COMMANDS = [
        ("vsftpd 누적 저장 로그 전체 분석", "cat /var/log/vsftpd.log")
    ]

    # 사용자 정의 SSH 클라이언트 생성
    client = get_ssh_client(ip, password, "root", 22)


        # 변경된 전용 run_cmd 실행
    run_cmd_for_audit(client, COMMANDS)

In [ ]:
import re
import paramiko

# ==========================================
# 1. 서버 접속 정보 설정
# ==========================================
HOST = "172.16.10.10"  # 서버 IP
PORT = 22
USER = "root"
PASSWORD = "asd123!@"  # 실제 비밀번호로 변경


# ==========================================
# 2. 원격 명령어 실행 공통 함수
# ==========================================
def execute_remote_cmd(cmd: str) -> str:
    """SSH 접속 후 단일 명령어를 실행하고 결과를 반환합니다."""
    ssh = paramiko.SSHClient()
    ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy())
    try:
        ssh.connect(HOST, port=PORT, username=USER, password=PASSWORD)
        stdin, stdout, stderr = ssh.exec_command(cmd)
        return stdout.read().decode("utf-8")
    except Exception as e:
        print(f"[!] 명령어 실행 실패: {e}")
        return ""
    finally:
        ssh.close()


# ==========================================
# 3. 로그 소스별 독립 함수 리스트
# ==========================================

def parse_last_logs(count=20):
    """last: 로그인 성공 이력 (wtmp)"""
    raw_data = execute_remote_cmd(f"last -n {count}")
    lines = raw_data.strip().split('\n')

    for line in lines:
        if not line.strip() or line.startswith("wtmp begins") or "reboot" in line:
            continue
        parts = line.split()
        if len(parts) >= 4:
            user = parts[0]
            tty = parts[1]
            ip = parts[2]
            time_str = " ".join(parts[3:7])
            remaining = " ".join(parts[7:])

            print("[SUCCESS LOGIN]")
            print(f"User    : {user}")
            print(f"Terminal: {tty}")
            print(f"IP/Host : {ip}")
            print(f"Time    : {time_str}")
            print(f"Duration: {remaining}")
            print("-" * 50)


def parse_lastb_logs(count=20):
    """lastb: 로그인 실패 이력 (btmp)"""
    raw_data = execute_remote_cmd(f"lastb -n {count}")
    lines = raw_data.strip().split('\n')

    for line in lines:
        if not line.strip() or line.startswith("btmp begins"):
            continue
        parts = line.split()
        if len(parts) >= 4:
            user = parts[0]
            tty = parts[1]
            ip = parts[2]
            time_str = " ".join(parts[3:7])

            print("[FAILED LOGIN]")
            print(f"User    : {user}")
            print(f"Terminal: {tty}")
            print(f"IP/Host : {ip}")
            print(f"Time    : {time_str}")
            print(f"Status  : Authentication Failure (공격 의심)")
            print("-" * 50)


def parse_who_logs():
    """who: 현재 접속자 목록"""
    raw_data = execute_remote_cmd("who")
    lines = raw_data.strip().split('\n')

    for line in lines:
        if not line.strip():
            continue
        parts = line.split()
        if len(parts) >= 5:
            user = parts[0]
            tty = parts[1]
            time_str = f"{parts[2]} {parts[3]}"
            ip = parts[4].strip("()")

            print("[CURRENT USER]")
            print(f"User    : {user}")
            print(f"Terminal: {tty}")
            print(f"Time    : {time_str}")
            print(f"IP      : {ip}")
            print("-" * 50)


def parse_lastlog_logs():
    """lastlog: 사용자별 최종 로그인 타임스탬프"""
    raw_data = execute_remote_cmd("lastlog")
    lines = raw_data.strip().split('\n')

    for line in lines[1:]: # 헤더 제외
        if not line.strip() or "Never logged in" in line:
            continue
        parts = line.split()
        if len(parts) >= 4:
            user = parts[0]
            # 터미널 글자 밀림 현상 방지 보정
            tty_part = parts[1]
            if "pts" in tty_part or "tty" in tty_part:
                tty = tty_part[:5]
                ip = tty_part[5:] if len(tty_part) > 5 else parts[2]
                time_idx = 3 if len(tty_part) > 5 else 2
            else:
                tty, ip, time_idx = "-", parts[1], 2

            time_str = " ".join(parts[time_idx:])

            print("[LASTLOG TIMESTAMP]")
            print(f"User    : {user}")
            print(f"Port    : {tty}")
            print(f"IP      : {ip}")
            print(f"Time    : {time_str}")
            print("-" * 50)


def parse_secure_filtered_logs(filter_type: str, lines_count=100):
    """
    /var/log/secure 관련 통합 필터 함수
    filter_type 매개변수: 'sudo', 'failed', 'accepted', 'useradd', 'userdel', 'passwd'
    """
    cmd_map = {
        "sudo": "grep 'sudo'",
        "failed": "grep 'Failed password'",
        "accepted": "grep 'Accepted'",
        "useradd": "grep -E 'useradd|new user'",
        "userdel": "grep 'userdel'",
        "passwd": "grep 'passwd'"
    }

    if filter_type not in cmd_map:
        print("[!] 잘못된 필터 타입입니다.")
        return

    raw_data = execute_remote_cmd(f"cat /var/log/secure | {cmd_map[filter_type]} | tail -n {lines_count}")
    lines = raw_data.strip().split('\n')

    pattern = r'^(?P<month>\w{3})\s+(?P<day>\d+)\s+(?P<time>\d{2}:\d{2}:\d{2}) (?P<host>\S+) (?P<service>\w+)(?:\[(?P<pid>\d+)\])?: (?P<msg>.*)'

    for line in lines:
        if not line.strip():
            continue
        m = re.match(pattern, line)
        if not m:
            continue

        data = m.groupdict()
        ip_match = re.search(r'(?:from|rhost=)(?P<ip>\d+\.\d+\.\d+\.\d+)', data['msg'])
        user_match = re.search(r'(?:for|user=)(?P<user>\w+)', data['msg'])

        print(f"[SECURE AUTH: {filter_type.upper()}]")
        print(f"Time    : 2026 {data['month']} {data['day']} {data['time']}")
        print(f"Host    : {data['host']}")
        print(f"Service : {data['service']} (PID: {data.get('pid', '-')})")
        print(f"User    : {user_match.group('user') if user_match else '-'}")
        print(f"IP      : {ip_match.group('ip') if ip_match else '-'}")
        print(f"Msg     : {data['msg']}")
        print("-" * 50)


def parse_audit_logs(lines_count=30):
    """/var/log/audit/audit.log: 커널 보안 감시 로그"""
    raw_data = execute_remote_cmd(f"cat /var/log/audit/audit.log | tail -n {lines_count}")
    lines = raw_data.strip().split('\n')

    for line in lines:
        if not line.strip():
            continue
        type_m = re.search(r'type=(?P<type>[A-Z_]+)', line)
        success_m = re.search(r'success=(?P<succ>\w+)', line)
        exe_m = re.search(r'exe="(?P<exe>[^"]+)"', line)
        key_m = re.search(r'key="(?P<key>[^"]+)"', line)
        ts_m = re.search(r'audit\((?P<ts>\d+)\.', line)

        print("[KERNEL AUDIT]")
        print(f"Type    : {type_m.group('type') if type_m else '-'}")
        print(f"Success : {success_m.group('succ') if success_m else '-'}")
        print(f"Program : {exe_m.group('exe') if exe_m else '-'}")
        print(f"WatchKey: {key_m.group('key') if key_m else '-'}")
        print(f"Raw Msg : {line[:120]}...")
        print("-" * 50)


def parse_cron_logs(lines_count=30):
    """/var/log/cron: 크론잡 실행 이력"""
    raw_data = execute_remote_cmd(f"cat /var/log/cron | tail -n {lines_count}")
    lines = raw_data.strip().split('\n')
    pattern = r'^(\w{3})\s+(\d+)\s+(\d+:\d+:\d+)\s+(\S+)\s+crond(?:\[\d+\])?: \((.*?)\) CMD \((.*)\)$'

    for line in lines:
        m = re.match(pattern, line)
        if m:
            mo, d, t, host, user, cmd = m.groups()
            print("[CRON JOB EXECUTION]")
            print(f"Time    : 2026 {mo} {d} {t}")
            print(f"Host    : {host}")
            print(f"User    : {user}")
            print(f"Command : {cmd}")
            print("-" * 50)


def parse_dnf_logs(lines_count=30):
    """/var/log/dnf.log 및 dnf.rpm.log 패키지 이력"""
    raw_data = execute_remote_cmd(f"cat /var/log/dnf.log | tail -n {lines_count}")
    lines = raw_data.strip().split('\n')

    for line in lines:
        if not line.strip():
            continue
        line_clean = line.replace("T", " ")
        parts = line_clean.split(maxsplit=2)
        if len(parts) >= 3:
            print("[DNF PACKAGE LOG]")
            print(f"Time    : {parts[0]} {parts[1][:8]}")
            print(f"Level   : {parts[1][9:].strip('[]')}")
            print(f"Msg     : {parts[2]}")
            print("-" * 50)


def parse_messages_logs(lines_count=30):
    """/var/log/messages: 시스템 전반 로그"""
    raw_data = execute_remote_cmd(f"cat /var/log/messages | tail -n {lines_count}")
    lines = raw_data.strip().split('\n')
    pattern = r"^([A-Z][a-z]{2})\s+(\d+)\s+(\d+:\d+:\d+)\s+(\S+)\s+(\S+?): (.*)$"

    for line in lines:
        m = re.match(pattern, line)
        if m:
            mo, d, t, host, svc, msg = m.groups()
            print("[SYSTEM MESSAGE]")
            print(f"Time    : 2026 {mo} {d} {t}")
            print(f"Host    : {host}")
            print(f"Service : {svc}")
            print(f"Message : {msg}")
            print("-" * 50)